In [ ]:
!pip install numpy matplotlib scipy

# Lecture 03: Bayesian Parametric Estimation

## Classification Problems and Bayesian Approach

## Big Idea
Bayesian estimation treats parameters as random variables with prior distributions, updating beliefs with evidence to form posterior distributions.

## Everyday Metaphor

> 💡 **Bayesian Estimation = updating beliefs with evidence**: Imagine you're trying to estimate the probability that a coin is heads. You start with a prior belief (maybe you think it's fair), then you flip the coin and observe outcomes. Each flip updates your belief about the coin's bias.
> 
> **Prior Distribution = initial belief**: Your starting assumption about parameters before seeing any data, expressed as a probability distribution.
> 
> **Likelihood = evidence from data**: How probable the observed data is for different parameter values.
> 
> **Posterior Distribution = updated belief**: Your revised belief after combining the prior with the observed data using Bayes' theorem.

## Why It Matters
- Provides a principled way to incorporate prior knowledge into statistical models
- Naturally handles uncertainty in parameter estimates
- Enables sequential learning as new data arrives
- Forms the foundation for Bayesian machine learning methods
- Essential for understanding modern approaches to model regularization and uncertainty quantification

## Formulas


### Bayes' Theorem

$$
P(\theta|\mathcal{D}) = \frac{P(\mathcal{D}|\theta) P(\theta)}{P(\mathcal{D})}
$$

*What each symbol means:*
- **P(\theta|\mathcal{D})**: Posterior probability of parameters \theta given data \mathcal{D}
- **P(\mathcal{D}|\theta)**: Likelihood of data \mathcal{D} given parameters \theta
- **P(\theta)**: Prior probability of parameters \theta
- **P(\mathcal{D})**: Marginal likelihood or evidence (normalizing constant)


### Beta-Binomial Conjugate Pair

$$
\text{Prior}: \quad P(\theta) = \text{Beta}(\theta; \alpha, \beta) = \frac{\theta^{\alpha-1}(1-\theta)^{\beta-1}}{B(\alpha,\beta)}
$$

$$
\text{Likelihood}: \quad P(\mathcal{D}|\theta) = \theta^k (1-\theta)^{n-k} \\ \text{(Binomial likelihood)}
$$

$$
\text{Posterior}: \quad P(\theta|\mathcal{D}) = \text{Beta}(\theta; \alpha+k, \beta+n-k)
$$

*What each symbol means:*
- **\theta**: Parameter representing probability of success
- **\alpha, \beta**: Prior parameters (pseudo-counts)
- **k**: Number of successes observed
- **n**: Total number of trials
- **B(\alpha,\beta)**: Beta function (normalizing constant)
- The beauty: posterior is same family as prior with updated parameters


### Posterior Predictive Distribution

$$
P(y_{new}=1|\mathcal{D}) = \int_0^1 \theta \, P(\theta|\mathcal{D}) \, d\theta = \frac{\alpha+k}{\alpha+\beta+n}
$$

*What each symbol means:*
- **P(y_{new}=1|\mathcal{D})**: Probability of success on next trial given observed data
- **\theta**: Parameter being integrated over (marginalized out)
- **P(\theta|\mathcal{D})**: Posterior distribution of \theta
- The integral computes the expected value of \theta under the posterior
- For Beta posterior: mean = \frac{\alpha}{\alpha+\beta}


## Worked Example by Hand

> 🧮 Coin flipping example: We observe 7 heads in 10 flips. Start with Beta(2,2) prior, find posterior and predictive distribution.

### Step 1: Specify the prior

We choose a Beta(2,2) prior, which represents a weak belief that the coin is fair.

$$
P(\theta) = \text{Beta}(\theta; 2, 2) = \frac{\theta^{2-1}(1-\theta)^{2-1}}{B(2,2)} = 6\theta(1-\theta)
$$

### Step 2: Write the likelihood

For 7 heads in 10 flips, the binomial likelihood is:

$$
P(\mathcal{D}|\theta) = \theta^7 (1-\theta)^{10-7} = \theta^7 (1-\theta)^3
$$

### Step 3: Compute the posterior (up to proportionality)

$$
P(\theta|\mathcal{D}) \propto P(\mathcal{D}|\theta) P(\theta) = [\theta^7 (1-\theta)^3] [6\theta(1-\theta)] = 6\theta^{8}(1-\theta)^{4}
$$

### Step 4: Recognize the posterior distribution

This is proportional to a Beta distribution:

$$
P(\theta|\mathcal{D}) = \text{Beta}(\theta; 2+7, 2+3) = \text{Beta}(\theta; 9, 5)
$$

### Step 5: Compute the posterior mean

$$
E[\theta|\mathcal{D}] = \frac{\alpha}{\alpha+\beta} = \frac{9}{9+5} = \frac{9}{14} \approx 0.643
$$

### Step 6: Compute the predictive distribution

$$
P(y_{new}=1|\mathcal{D}) = \frac{\alpha+k}{\alpha+\beta+n} = \frac{2+7}{2+2+10} = \frac{9}{14} \approx 0.643
$$

**Answer: After observing 7 heads in 10 flips with Beta(2,2) prior, the posterior is Beta(9,5) and the predicted probability of heads on the next flip is 9/14 ≈ 0.643.**

<em>This demonstrates how Bayesian estimation combines prior beliefs with observed data.</em>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


In [ ]:
# Define parameters for our example
prior_alpha, prior_beta = 2, 2
observed_heads = 7
total_flips = 10

# Compute posterior parameters
posterior_alpha = prior_alpha + observed_heads
posterior_beta = prior_beta + (total_flips - observed_heads)

print(f"Prior: Beta({prior_alpha}, {prior_beta})")
print(f"Likelihood: {observed_heads} heads in {total_flips} flips")
print(f"Posterior: Beta({posterior_alpha}, {posterior_beta})")
print(f"Posterior mean: {posterior_alpha/(posterior_alpha+posterior_beta):.4f}")


In [ ]:
# Create visualization
theta = np.linspace(0, 1, 1000)

# Compute distributions
prior = stats.beta.pdf(theta, prior_alpha, prior_beta)
likelihood = stats.beta.pdf(theta, observed_heads+1, total_flips-observed_heads+1)  # For visualization
likelihood = likelihood / np.max(likelihood)  # Normalize for plotting
posterior = stats.beta.pdf(theta, posterior_alpha, posterior_beta)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(theta, prior, 'b-', linewidth=2, label=f'Prior: Beta({prior_alpha}, {prior_beta})')
plt.plot(theta, likelihood, 'g--', linewidth=2, label='Likelihood (normalized)')
plt.plot(theta, posterior, 'r-', linewidth=2, label=f'Posterior: Beta({posterior_alpha}, {posterior_beta})')
plt.xlabel('$\theta$ (probability of heads)')
plt.ylabel('Probability Density')
plt.title('Bayesian Coin Flipping: Prior, Likelihood, and Posterior')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axvline(x=observed_heads/total_flips, color='k', linestyle=':', alpha=0.7, label=f'MLE: {observed_heads/total_flips:.2f}')
plt.legend()
plt.show()


## Bayesian Concepts for Classification

> 📝 Extending Bayesian estimation to classification problems.

### Naive Bayes Classifier

For binary classification with features $x_1, x_2, \dots, x_n$:

$$
P(y=1|\mathbf{x}) = \frac{P(\mathbf{x}|y=1) P(y=1)}{P(\mathbf{x})}
$$

Under the naive assumption of feature independence:

$$
P(\mathbf{x}|y) = \prod_{i=1}^n P(x_i|y)
$$

### Parameter Estimation

- **Class priors**: $P(y=c) = \frac{N_c}{N}$ (frequency of class c)
- **Feature likelihoods**: For binary features, $P(x_i=1|y=c) = \frac{N_{ic}}{N_c}$
- **Laplace smoothing**: Add pseudo-counts to avoid zero probabilities


In [ ]:
# Simple Naive Bayes implementation for binary features
import numpy as np

class NaiveBayes:
    def __init__(self, alpha=1.0):  # Laplace smoothing parameter
        self.alpha = alpha
        self.class_priors = None
        self.feature_probs = None
        self.classes = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        # Initialize arrays
        self.class_priors = np.zeros(n_classes)
        self.feature_probs = np.zeros((n_classes, n_features))
        
        # Calculate class priors and feature probabilities
        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.class_priors[idx] = (len(X_c) + self.alpha) / (n_samples + n_classes * self.alpha)
            
            # For each feature, compute P(x_i=1|y=c) with Laplace smoothing
            for i in range(n_features):
                count = np.sum(X_c[:, i] == 1)
                self.feature_probs[idx, i] = (count + self.alpha) / (len(X_c) + 2 * self.alpha)

    def predict_proba(self, X):
        n_samples = X.shape[0]
        probs = np.zeros((n_samples, len(self.classes)))
        
        for i, c in enumerate(self.classes):
            # log P(y=c) + sum log P(x_i|y=c)
            log_prior = np.log(self.class_priors[i])
            log_likelihood = np.sum(
                X * np.log(self.feature_probs[i]) + 
                (1 - X) * np.log(1 - self.feature_probs[i]), 
                axis=1
            )
            probs[:, i] = log_prior + log_likelihood
        
        # Convert log probabilities to probabilities
        probs = np.exp(probs - np.max(probs, axis=1, keepdims=True))  # For numerical stability
        probs = probs / np.sum(probs, axis=1, keepdims=True)
        return probs

    def predict(self, X):
        return self.classes[np.argmax(self.predict_proba(X), axis=1)]


In [ ]:
# Test Naive Bayes on a simple dataset
# Create synthetic data: 2 features, binary classification
np.random.seed(42)
n_samples = 100

# Class 0: features mostly 0
X0 = np.random.binomial(1, 0.2, (n_samples//2, 2))
# Class 1: features mostly 1
X1 = np.random.binomial(1, 0.8, (n_samples//2, 2))
X = np.vstack([X0, X1])
y = np.hstack([np.zeros(n_samples//2), np.ones(n_samples//2)])

# Shuffle the data
indices = np.random.permutation(n_samples)
X = X[indices]
y = y[indices]

# Split into train/test
train_size = int(0.8 * n_samples)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Train Naive Bayes
nb = NaiveBayes(alpha=1.0)
nb.fit(X_train, y_train)

# Make predictions
y_pred = nb.predict(X_test)
y_pred_proba = nb.predict_proba(X_test)

# Calculate accuracy
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy:.4f}")

# Show some predictions
print("\nSample predictions:")
for i in range(min(5, len(X_test))):
    print(f"True: {y_test[i]}, Pred: {y_pred[i]}, Prob: [{y_pred_proba[i][0]:.3f}, {y_pred_proba[i][1]:.3f}]")


## Watch Out: Common Mistakes & Exam Traps

> ⚠️ **Common mistakes to avoid:**
> 1. **Confusing likelihood with probability**: Likelihood is not a probability distribution over parameters; it's a function of parameters given fixed data.
> 2. **Ignoring the normalizing constant**: In Bayesian inference, the posterior is proportional to likelihood × prior, but you often need the normalizing constant for model comparison.
> 3. **Choosing inappropriate priors**: Priors should reflect genuine prior knowledge or be weakly informative; improper priors can lead to improper posteriors.
> 4. **Misinterpreting credible intervals**: A 95% credible interval means there's a 95% probability that the parameter lies in that interval (given the data and prior), not that 95% of such intervals from repeated experiments would contain the true parameter.
> 5. **Overlooking model complexity**: More complex models can fit data better but may overfit; Bayesian model comparison automatically penalizes complexity through the marginal likelihood.


## Experiment Link

> 🔬 Run **Section 2: Bayesian Coin Flipping** in this notebook to visualize how different priors affect the posterior when observing coin flip data.

This section will include:
- Interactive exploration of different Beta priors
- Visualization of prior, likelihood, and posterior distributions
- Calculation of posterior statistics and predictive distributions
- Sequential updating as new data arrives


## Videos to Watch

> 📺 **Recommended video searches:**

1. **Bayesian Statistics for Beginners**
   - Search: https://www.youtube.com/results?search_query=Bayesian+statistics+beginners
   - Watch for: Intuitive explanation of Bayes' theorem and prior/posterior concepts

2. **StatQuest: Naive Bayes**
   - Search: https://www.youtube.com/results?search_query=StatQuest+Naive+Bayes
   - Watch for: Clear explanation of how Naive Bayes works and when to use it

3. **Mathematical Monk: Bayesian Inference**
   - Search: https://www.youtube.com/results?search_query=Mathematical+Monk+Bayesian+inference
   - Watch for: Mathematical treatment of Bayesian concepts with examples


## Practice Questions

> ❓ Try to solve these yourself first, then check the answers below!

### Question 1 (Concept): What is the key difference between frequentist and Bayesian approaches to probability?

<details>
<summary>Click to reveal answer</summary>

Frequentists interpret probability as long-run frequency of events, while Bayesians interpret probability as a degree of belief or uncertainty that can be updated with new evidence.

</details>


### Question 2 (Concept): What does it mean for a prior to be conjugate to a likelihood?

<details>
<summary>Click to reveal answer</summary>

A prior is conjugate to a likelihood if the posterior distribution belongs to the same family as the prior distribution, making analytical computation of the posterior possible.

</details>


### Question 3 (Calculation): If we have a Beta(3,3) prior and observe 5 heads in 8 flips, what is the posterior distribution?

<details>
<summary>Click to reveal answer</summary>

Posterior = Beta(3+5, 3+(8-5)) = Beta(8, 6)

</details>


### Question 4 (Concept): Why do we add pseudo-counts in Laplace smoothing for Naive Bayes?

<details>
<summary>Click to reveal answer</summary>

Laplace smoothing adds pseudo-counts to avoid zero probabilities, which would cause the entire posterior probability to become zero when multiplying likelihoods (since anything times zero is zero).

</details>


### Question 5 (Explain it simply): When would you prefer a Bayesian approach over a frequentist approach?

<details>
<summary>Click to reveal answer</summary>

Prefer Bayesian approaches when: you have prior knowledge to incorporate, you need to quantify uncertainty in parameters, you're working with small datasets, or you want to perform sequential learning as new data arrives.

</details>


## Cheat Sheet Draft

> 📝 This section is optimized for printing - it should fit approximately 1/7 of an A4 page when printed.

- Optimized for printing - fits approximately 1/7 of an A4 page
- Contains key formulas, definitions, and common mistakes
- Designed for quick review and reference

### Key Formulas

$$
P(\theta|\mathcal{D}) = \frac{P(\mathcal{D}|\theta) P(\theta)}{P(\mathcal{D})}
$$

$$
P(\theta|\mathcal{D}) = \text{Beta}(\theta; \alpha+k, \beta+n-k) \\ \text{(for Beta-Binomial)}
$$

$$
P(y_{new}=1|\mathcal{D}) = \frac{\alpha+k}{\alpha+\beta+n}
$$

### Key Definitions

- **Prior Distribution:** Initial belief about parameters before seeing data
- **Likelihood:** Probability of observing the data given specific parameter values
- **Posterior Distribution:** Updated belief about parameters after seeing data
- **Conjugate Prior:** Prior that results in posterior from same family when combined with likelihood
- **Credible Interval:** Bayesian analog to confidence interval; interval containing specified probability mass

### Common Mistakes to Avoid

- Confusing likelihood with probability
- Ignoring the normalizing constant in Bayes' theorem
- Choosing inappropriate priors that bias results
- Misinterpreting credible intervals as frequentist confidence intervals
- Overlooking model complexity in Bayesian model comparison
